# CPA attack on standard ASCON software implementation

This notebook performs the **CPA attack** on the previously acquired power traces.  
It loads the traceset produced by the acquisition notebook ( the h5 file (.h5)) and computes:

- **Correlation Power Analysis (CPA)** results  
- **Key rank** vs. number of traces  
- **Correlation trends** for each key byte, comparing correct vs. wrong key hypotheses  

These plots help evaluate the **difficulty of recovering each key byte** and visualize the leakage behavior across the trace window.

⚠️ **Important:**  
This notebook assumes that the **Power Trace Acquisition** notebook (`xheep_capture_ASCON.ipynb`) has already been executed and the traceset has been acquired.


In [ ]:
import sys
import os
import time
from pathlib import Path

## Project paths
This block robustly detects the project root (`DOJO_ROOT`) starting from either the script location (`__file__`) or the current working directory (for notebooks). From there it defines all relevant subdirectories (ASCON sources, SCA scripts, X-HEEP, traces, plots, cache), ensures output folders exist, and adds the local source paths to `sys.path` .

In [ ]:
def find_project_root(start: Path, markers=("fusesoc.conf", ".dojo_root")) -> Path:
    current = start
    while current != current.parent:
        if any((current / m).exists() for m in markers):
            return current
        current = current.parent

    raise RuntimeError(
        f"Could not find project root (looked for markers: {markers}). "
        "Please ensure you are inside the Side-Channel-Dojo repository."
    )


# In a script, __file__ exists; in a notebook, it does not.
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    # Notebook / interactive: use the current working directory instead
    SCRIPT_DIR = Path.cwd()

DOJO_ROOT = find_project_root(SCRIPT_DIR)

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
ASCON_PY_DIR = DOJO_ROOT / "sw" / "ciphers" / "ASCON_init_python"
SCA_DIR      = DOJO_ROOT / "sw" / "sca_scripts"
HW_DIR       = DOJO_ROOT / "hw"

# Traces and plots PATH for ASCON SW
TRACESET_DIR = DOJO_ROOT / "sw" / "traceset" / "ASCON" / "sw"
BASE_PLOT_DIR = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "plot"
BASE_CACHE_DIR      = DOJO_ROOT / "sw" / "sca_scripts" / "ASCON" / "sw" / "cache" 

BASE_PLOT_DIR.mkdir(parents=True, exist_ok=True)
BASE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Make local modules importable without fragile ../ relative paths
sys.path.insert(0, str(ASCON_PY_DIR))
sys.path.insert(0, str(SCA_DIR))
sys.path.insert(0, str(BASE_CACHE_DIR))

## Imports

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import logging
import json
import h5py

# Configuration

In [ ]:
# ---------------------------------------------------------------------------
# ASCON S-box configuration
# ---------------------------------------------------------------------------

# Available S-box implementations
AVAILABLE_SBOXES = [
    "lut_ascon",
    "lut_bilgin",
    "lut_allouzi",
    "lut_lu_4",
    "lut_lu_5",
    "lut_lu_6",
    "lut_lu_7",
]

# Select which ASCON S-box / substitution layer implementation to test
sbox_type = "lut_lu_5"   # choose from AVAILABLE_SBOXES
assert sbox_type in AVAILABLE_SBOXES, f"Unknown S-box type: {sbox_type}"
tested_sbox = sbox_type  # alias used later in the printout

# ---------------------------------------------------------------------------
# Flow flags
# ---------------------------------------------------------------------------

# Traces are assumed to be already captured; we just load them and analyse.
n_trc                  = 1_000_000  # Total number of traces in the traceset
traces_overlapped_plot = True       # Plot overlapped power traces
save_plot              = True       # Save overlapped trace plot to disk

# CPA / analysis cache control
load_attack_results = True   # Load CPA cache if available
save_attack_results = True   # Save CPA results to cache after the run

# Plot control
key_rank_plot         = True   # Plot PGE vs traces
traces_correlation_plot = True   # Plot correlation vs traces

# Output control
save_plots   = True   # Save plots to disk
save_results = True   # Save analysis results (JSON, etc.) to disk

# ---------------------------------------------------------------------------
# traceset file, directories
# ---------------------------------------------------------------------------

# Traces file path (board-captured traces)
# Example filename: ascon_opt32_lut_lu_5_1000k_tmp.h5 for n_trc = 1_000_000
TRACESET_FILE = TRACESET_DIR / f"ascon_opt32_{sbox_type}_{n_trc // 1000}k_tmp.h5"

CACHE_DIR      = BASE_CACHE_DIR / sbox_type
PLOT_DIR       = BASE_PLOT_DIR / sbox_type
CPA_CACHE_FILE = CACHE_DIR / f"CPA_results_{sbox_type}.json"

# Ensure traceset and plot directories exist
TRACESET_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# Configuration printout
# ---------------------------------------------------------------------------

def _yn(flag: bool) -> str:
    """Return 'yes' or 'no' for a boolean flag."""
    return "yes" if flag else "no"


print("\n================= CONFIGURATION =================")
print(f"DOJO_ROOT           : {DOJO_ROOT}")
print()
print("Target")
print(f"  Tested S-box              : {tested_sbox}")
print()
print("Paths")
print(f"  Traceset file             : {TRACESET_FILE}")
print(f"  Plot dir                  : {PLOT_DIR}")
print(f"  CPA cache file            : {CPA_CACHE_FILE}")
print()
print("Analysis configuration")
print(f"  Total traces (n_trc)      : {n_trc}")
print(f"  Save plots                : {_yn(save_plots)}")
print(f"  Save results              : {_yn(save_results)}")
print(f"  Load CPA results (cache)  : {_yn(load_attack_results)}")
print(f"  Save CPA results (cache)  : {_yn(save_attack_results)}")
print()
print(f"  Key rank plot             : {_yn(key_rank_plot)}")
print(f"  Correlation plot          : {_yn(traces_correlation_plot)}")
print("=================================================\n")

### CPA Attack on ASCON with 3-bit Selection Function

This notebook implements a **Correlation Power Analysis (CPA)** attack on ASCON using a **3-bit selection function** as leakage model. The attack targets a **single bit** of an internal state register after the **first-round substitution + linear diffusion layer**, and tries to recover **3 key bits** at a time.

The selection function and the attacked structure are illustrated in the figure below:

![ASCON selection function](media/ASCON_selection_function.png)


---

#### 1. Target and Leakage Model

- We focus on **one state register bit**:
  - `state_register_index = 0` → register \(x_0\)
  - `bit_index` → attacked bit position \(j\) in \(x_0\) (e.g. `bit_index = 60`)
- For each nonce, the leakage model:
  1. Computes the internal bit at the **output of the linear diffusion layer** in round 1.
  2. Uses the **3-bit key selection function** shown in the figure  
     (e.g. bits \(($k_j$, $k_{j+19}$, $k_{j+28}$)\) for `x0`).
- The leakage model returns **8 hypotheses** (for key guesses `0..7`), each predicting the value of the attacked bit for that nonce.

Concretely:

- `ascon_leakage_model(...)` takes:
  - `init_vect`, `nonce_MSB`, `nonce_LSB`, `state_register_index`, `bit_index`, `sbox_type`
  - and (optionally) `key_0` when attacking `x1`
- It returns a vector of length 8:  
  $\text{leakage\_model}[k] \in \{0,1\}, \quad k = 0..7$

  where each entry is the predicted bit at the diffusion-layer output given 3-bit key guess `k`.

---

#### 2. Building the Hypothetical Leakage Matrix \(H\)

For a given number of traces \(N\) (for example, $N = \text{count} \times \text{resolution}$):

1. We have:
   - `partial_traces` → shape \((N, M)\): \(N\) traces, \(M\) time samples each.
   - `partial_nonces` → list/array of length \(N\), each with MSB/LSB components.

2. For each trace index \(n = 0..N-1\):
   - Extract `nonce_MSB`, `nonce_LSB` from `partial_nonces[n]`.
   - Call `ascon_leakage_model(...)` to get the 8 predicted bits for that nonce.
   - Store them in row `n` of `H_matrix`.

Result:

- `H_matrix` has shape \((N, 8)\):
  - columns correspond to **key hypotheses** \(k = 0..7\),
  - rows correspond to **traces / nonces**.

---

#### 3. Correlation Power Analysis (CPA)

The function `ascon_cpa(traces, hypothetical_values)` performs the actual CPA:

- Inputs:
  - `traces` → \((N, M)\)
  - `hypothetical_values` → \((N, K)\), here \(K = 8\)
- For each key hypothesis \(k\) and each sample index \(t\):
  - Take:
    - `x = traces[:, t]` → power samples across traces at time \(t\)
    - `y = hypothetical_values[:, k]` → predicted leakage for hypothesis \(k\)
  - Compute the Pearson correlation:
    \[
    R[k, t] = \text{corr}(x, y)
    \]
  - If inputs contain NaN/Inf or have zero variance, store `0.0` as a safe fallback.

Output:

- `R_matrix` of shape \((K, M)\), where:
  - row = key guess,
  - column = time sample index.

For the attack we use:

- `np.max(np.abs(R_matrix), axis=1)` → for each key guess \(k\), take the **maximum absolute correlation** over all samples \(t\).  
  This gives a vector `corr_vs_keyguess` of length 8.

---

#### 4. Incremental Traces: Correlation vs Number of Traces

To study how many traces are needed to distinguish the correct key guess, the notebook repeats the CPA with an increasing number of traces:

1. Choose a **step size** `resolution` (e.g. 5000).
2. For each `count = 1, 2, …`:
   - Use the first `count * resolution` traces and nonces.
   - Build `H_matrix` for these traces.
   - Run `ascon_cpa` to get `R_matrix` and `corr_vs_keyguess`.
   - Store `corr_vs_keyguess` in a list.

At the end:

- `corr_vs_traces` has shape `(num_steps, 8)`:
  - each row corresponds to a given number of traces,
  - each column to a particular 3-bit key guess.

We then plot **correlation vs number of traces** for all 8 hypotheses:

- x-axis: number of traces (`count * resolution`)
- y-axis: maximum absolute correlation for each key guess
- The **correct 3-bit key** should gradually produce the **highest correlation curve**, separating from wrong guesses as the number of traces increases.

This implements a **single-bit, 3-bit key-guess CPA** powered by the selection function in `media/ASCON_selection_function.png`, applied repeatedly for growing trace counts to evaluate attack performance.


# Loading the traceset file

In [ ]:
try:
    with h5py.File(TRACESET_FILE, "r") as f_read_traces:
        # Basic sanity: check that required datasets exist
        if "traces" not in f_read_traces or "nonces" not in f_read_traces:
            raise KeyError(
                "HDF5 file is missing required datasets 'traces' and/or 'nonces'."
            )

        total_traces = f_read_traces["traces"].shape[0]
        # Use at most N traces, but do not exceed what's in the file
        n_used = min(n_trc, total_traces)

        # Use slicing so data are actually loaded into RAM
        traces = f_read_traces["traces"][:n_used]
        nonces = f_read_traces["nonces"][:n_used]

    # Sanity check: traces and nonces should have the same number of rows
    if traces.shape[0] != nonces.shape[0]:
        raise ValueError(
            f"Number of traces ({traces.shape[0]}) and nonces ({nonces.shape[0]}) "
            "do not match. Check the traces file."
        )

    print(f"[INFO] Loaded {traces.shape[0]} traces from {TRACESET_FILE}")

except FileNotFoundError:
    print(f"[ERROR] Traces file {TRACESET_FILE} not found. "
          "Please run the trace acquisition phase first.")
except Exception as e:
    print(f"[ERROR] Could not read traces file {TRACESET_FILE}: {e}")


sw/notebook/examples/ascon/media/ASCON_selection_function.png

### `ascon_leakage_model(...)` (3-bit / 8-hypothesis version)

Builds a **CPA leakage model for ASCON** targeting **one bit** of the state *after* the first-round **substitution + linear diffusion layer**, under the assumption that **only 3 key bits are unknown**.

It returns **8 hypothetical leakages** (for key guesses `0..7`), each being the predicted value (0/1) of the attacked bit for that key guess.

**Inputs**

- `init_vect` *(int)* – IV, initial value of state register `x0`.
- `nonce_MSB` *(int)* – most significant 64 bits of the nonce (`x3`).
- `nonce_LSB` *(int)* – least significant 64 bits of the nonce (`x4`).
- `state_register_index` *(int)* – attacked state register (`0` or `1`).
- `bit_index` *(int)* – attacked bit position (`0..63`) in the chosen register.
- `sbox_type` *(str)* – S-box implementation identifier (e.g. `"lut_ascon"`, `"lut_bilgin"`, …).
- `key_0` *(int, optional)* – most significant 64 bits of the key; used when attacking `x1`, assuming `x0` has already revealed these bits.

**Output**

- `leakage_model` *(np.ndarray, shape `(8,)`, dtype `uint8`)*  
  For each **3-bit key guess** `k = 0..7`, `leakage_model[k]` is the predicted value (0/1) of the attacked bit at the output of the linear diffusion layer in the first round.

**How it works (brief)**

1. Defines:
   - `ascon_substitution_layer(...)` to compute the 5-bit S-box output for a given bit position (with row shifts and round constant xor on `x2`).
   - `ascon_shift_layer(...)` to combine three S-box outputs (with different row shifts) via XOR, modelling the **linear diffusion layer output** for that bit.

2. For each key guess `k = 0..7`:
   - If `state_register_index == 0` (attacking `x0`):
     - Interpret `k` as 3 unknown bits of the **MSB half of the key** (`key_guess_0`).
     - Fix the 3 bits of the **LSB half** (`key_guess_1`) to `0`, as assumed in the paper.
     - Compute the diffusion-layer output `Z` and take its MSB bit `Z_0 = (Z >> 4) & 1` as the hypothetical leakage.
   - If `state_register_index == 1` (attacking `x1`):
     - Derive 3 bits of the MSB half from the already-known `key_0` at positions shifted by `row_shift_1` (these are considered recovered from the attack on `x0`).
     - Interpret `k` as the 3 unknown bits of the **LSB half of the key** (`key_guess_1`).
     - Compute `Z` and take the next bit `Z_1 = (Z >> 3) & 1` as the hypothetical leakage.

3. Store the predicted bit in `leakage_model[k]` and return the array.


In [ ]:
if cpa_phase_full_key:

            # Full key recovery phase
            print("Starting full key recovery phase...")

            # Print the number of traces and the number of samples for each trace
            print(f"Number of traces: {traces.shape[0]}")
            print(f"Number of samples per trace: {traces.shape[1]}")
            print(f"S-box type: {sbox_type}\n")

            # Mapping of sbox_type to key bit indexes
            key_bit_indexes_0_dict = {
                "lut_ascon": [32, 13, 34, 4, 6, 54, 36, 0, 33, 63, 7, 16, 55, 19, 17, 41, 1, 40, 8, 48, 24, 39, 14, 31, 58, 49, 56, 47, 37, 29, 15, 46, 57, 11],
                "lut_bilgin": [4, 51, 7, 63, 31, 40, 32, 3, 43, 23, 59, 16, 13, 47, 36, 0, 41, 34, 44, 33, 6, 54, 48, 19, 17, 1, 10, 39, 56, 60, 18, 38, 11, 57, 49],
                "lut_lu_7": [32, 0, 13, 4, 16, 33, 15, 34, 63, 54, 55, 43, 6, 31, 14, 7, 39, 36, 17, 40, 48, 1, 19, 41, 24, 11, 3, 47, 29, 27, 37, 57, 8, 49],
                "lut_lu_6": [32, 13, 4, 63, 36, 33, 54, 34, 62, 16, 14, 0, 6, 17, 19, 43, 39, 40, 1, 55, 41, 8, 48, 47, 30, 58, 31, 56, 60, 38, 37, 57, 18],
                "lut_lu_5": [60, 30, 61, 28, 32, 0, 34, 23, 53, 14, 22, 44, 33, 5, 17, 38, 13, 62, 8, 56, 57, 6, 31, 15, 37, 10, 12, 39, 29, 36, 54, 49, 46, 35, 43],
                "lut_lu_4": [32, 13, 4, 36, 33, 54, 34, 6, 16, 14, 63, 0, 17, 19, 39, 43, 40, 1, 7, 55, 24, 41, 47, 48, 29, 58, 37, 50, 8, 12, 18, 57, 15, 49, 30],
                "lut_allouzi": [32, 33, 34, 14, 13, 4, 36, 30, 1, 22, 53, 31, 0, 29, 44, 62, 17, 47, 9, 54, 41, 63, 37, 19, 5, 46, 24, 16, 20, 60, 7, 2, 40, 61, 42, 39, 57],
            }
            key_bit_indexes_1_dict = {
                "lut_ascon": [32, 0, 63, 1, 14, 13, 36, 15, 31, 8, 38, 43, 5, 18, 23, 12, 45, 16, 9, 42, 3, 51, 2, 49, 24, 20, 44, 40, 28, 30, 37, 19, 47, 59, 53, 4, 46],
                "lut_bilgin": [32, 0, 63, 1, 36, 14, 13, 12, 31, 11, 45, 60, 62, 47, 41, 52, 8, 33, 46, 20, 48, 54, 44, 18, 61, 34, 58, 4, 24, 28, 26, 5, 59],
                "lut_lu_7": [50, 8, 42, 11, 59, 60, 17, 32, 49, 0, 63, 3, 10, 28, 43, 36, 1, 56, 34, 18, 33, 27, 4, 13, 25, 9, 6, 20, 19, 23, 5, 12, 15, 2, 62, 46, 55, 29],
                "lut_lu_6": [50, 8, 32, 0, 63, 1, 60, 36, 3, 56, 14, 13, 4, 28, 31, 9, 6, 12, 42, 23, 59, 19, 15, 30, 43, 44, 52, 2, 46, 49, 40, 51, 55, 22],
                "lut_lu_5": [50, 8, 32, 0, 63, 60, 1, 36, 6, 14, 4, 13, 31, 9, 42, 59, 16, 18, 5, 33, 44, 48, 15, 19, 40, 12, 20, 10, 46, 49, 30, 22, 29],
                "lut_lu_4": [32, 0, 1, 63, 47, 31, 36, 60, 8, 4, 3, 9, 56, 37, 35, 16, 2, 7, 6, 30, 11, 26, 52, 12, 28, 54, 19, 62, 15, 43, 20, 27, 46, 14],
                "lut_allouzi": [32, 0, 63, 1, 14, 36, 13, 31, 8, 38, 3, 9, 45, 18, 12, 30, 48, 15, 19, 43, 24, 46, 44, 40, 20, 2, 26, 55, 4, 37, 28, 11, 49, 59, 47],
            }

            # Select the correct list based on sbox_type
            try:
                key_bit_indexes_0 = key_bit_indexes_0_dict[sbox_type]
                key_bit_indexes_1 = key_bit_indexes_1_dict[sbox_type]
            except KeyError:
                raise ValueError(f"Unknown sbox_type: {sbox_type}")

            k0_bits = np.zeros(64, dtype=np.uint8)
            k1_bits = np.zeros(64, dtype=np.uint8)

            tic = time.perf_counter()
            print()

            # Iterate over all key bits of k0
            for key_bit in tqdm(key_bit_indexes_0, "Key bit recovery progress"):
                # Build the leakage model matrix
                H_matrix = np.empty((len(nonces), 8), dtype=np.uint8)
                R_matrix = np.empty((8, traces.shape[1]), dtype=np.float64)

                for n in range(len(nonces)):
                    nonce_MSB = nonces[n][1]
                    nonce_LSB = nonces[n][0]
                    leakage_model_i = ascon_leakage_model(initialization_vector, nonce_MSB, nonce_LSB, 0, key_bit, sbox_type)
                    H_matrix[n] = leakage_model_i

                # CPA attack
                R_matrix = ascon_cpa(traces, H_matrix)

                # Find the time samples with the maximum correlation value
                corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1) # shape (8,)

                # Find the key guess with the maximum correlation value
                best_key_guess = np.argmax(corr_vs_keyguess)

                # Save the result to the k0 vector
                k0_bits[(key_bit)      % 64] = (best_key_guess >> 2) & 1
                k0_bits[(key_bit + 19) % 64] = (best_key_guess >> 1) & 1
                k0_bits[(key_bit + 28) % 64] = (best_key_guess >> 0) & 1

            # Convert the numpy array into a single hex integer
            k0 = 0
            for i in range(64):
                k0 |= ((k0_bits[i] & 0x01) << i)
            k0 = int(k0) & 0xFFFFFFFFFFFFFFFF
            print(f"Recovered most significand half of the key: {k0:016x}")
            print()

            # Iterate over all key bits of k1
            for key_bit in tqdm(key_bit_indexes_1, "Key bit recovery progress"):
                # Build the leakage model matrix
                H_matrix = np.empty((len(nonces), 8), dtype=np.uint8)
                R_matrix = np.empty((8, traces.shape[1]), dtype=np.float64)

                for n in range(len(nonces)):
                    nonce_MSB = nonces[n][1]
                    nonce_LSB = nonces[n][0]
                    leakage_model_i = ascon_leakage_model(initialization_vector, nonce_MSB, nonce_LSB, 1, key_bit, sbox_type, k0)
                    H_matrix[n] = leakage_model_i

                # CPA attack
                R_matrix = ascon_cpa(traces, H_matrix)

                # Find the time samples with the maximum correlation value
                corr_vs_keyguess = np.max(np.abs(R_matrix), axis=1) # shape (8,)

                # Find the key guess with the maximum correlation value
                best_key_guess = np.argmax(corr_vs_keyguess)

                # Save the result to the k1 vector
                k1_bits[(key_bit)      % 64] = (best_key_guess >> 2) & 1
                k1_bits[(key_bit + 61) % 64] = (best_key_guess >> 1) & 1
                k1_bits[(key_bit + 39) % 64] = (best_key_guess >> 0) & 1

            # Convert the numpy array into a single hex integer. The XOR operation is needed to isolate the k1 bits
            k1 = 0
            for i in range(64):
                k1 |= (k1_bits[i] & 0x01) << i
            k1 = int(k1) & 0xFFFFFFFFFFFFFFFF
            # The key recovery from the register z1 actually need this extra XOR operation
            #k1 = k1 ^ k0
            print(f"Recovered least significand half of the key: {k1:016x}\n")

            print(f"Recovered full key: {k1:016x}{k0:016x}")

            # Convert to hex strings
            k0_hex = f"{k0:016X}"
            k1_hex = f"{k1:016X}"
            # Combine the two halves
            recovered_key = k1_hex + k0_hex
            # Reverse both key strings by groups of 2 char and then reverse
            recovered_key = [recovered_key[i:i + 2] for i in range(0, len(recovered_key), 2)][::-1]
            recovered_key = ''.join(recovered_key)

            print(f"Recovered key (little-endian): 0x{recovered_key}")

            if recovered_key != key_for_printing:
                print(f"\033[91mERROR\033[0m: Key recovery failed.\nGot: 0x{recovered_key}\nExpected: 0x{key_for_printing}\n")
            else:
                print("\033[92mSUCCESS\033[0m: Key correctly recovered!\n")

            toc = time.perf_counter()
            print(f"Full key recovery phase completed in {(toc - tic)/60:.2f} minutes.")

### `ascon_leakage_model(...)`

Builds the **leakage model used for a CPA attack on ASCON**, targeting **one bit** of the state *after* the first-round **substitution + linear diffusion layer**.

For each of the \(2^6 = 64\) hypotheses on 6 key bits, it predicts the value of the attacked bit and returns these predictions as a NumPy array.

**Inputs**

- `init_vect` *(int)* – IV, initial value of state register `x0`.
- `nonce_MSB` *(int)* – most significant 64 bits of the nonce (`x3`).
- `nonce_LSB` *(int)* – least significant 64 bits of the nonce (`x4`).
- `state_register_index` *(int)* – attacked state register (0 or 1).
- `bit_index` *(int)* – attacked bit position in the register (`0..63`).
- `sbox_type` *(str)* – S-box implementation identifier (e.g. `"lut_ascon"`).
- `key_0` *(int, optional)* – MSB half of the key (reserved for scenarios where part of the key is already known).

**Output**

- `leakage_model` *(np.ndarray, shape `(64,)`, dtype `uint8`)*  
  - For each 6-bit key guess `k = 0..63`, `leakage_model[k]` is the predicted value (0/1) of the attacked bit at the output of the linear diffusion layer in the first round.

**How it works (brief)**

1. For each key guess (`0..63`), split the 6 bits into:
   - 3 bits for the MSB half of the key,  
   - 3 bits for the LSB half.
2. For the selected `bit_index`, compute the corresponding 5-bit S-box inputs from registers `x0..x4`, including row shifts and round constant injection, and obtain S-box outputs via `ascon.sbox(sbox_type, ...)`.
3. Combine three S-box outputs according to the ASCON linear diffusion layer to get a 5-bit word `Z`.
4. Take the MSB of `Z` as the hypothetical leakage bit and store it in `leakage_model[k]`.

This array can then be used as the **hypothetical leakage** in the CPA (correlation) step.
